In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator before running."
print(torch.cuda.get_device_name(0))


In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/solar-fil")
revision = "origin/main"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/Dharun235/kaggle-solar-filament-segmentation.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", revision], check=True)
os.chdir(repo)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
import shlex
import sys

run_dir = Path("/kaggle/working/artifacts/runs/unet_pq_v3")
data_root = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
model_command = shlex.join([
    sys.executable, "models/patch_unet.py",
    "--train", "{train_manifest}", "--val", "{val_manifest}", "--test", "{test_manifest}",
    "--raw-val", "{raw_val}", "--raw-test", "{raw_test}", "--run-dir", "{run_dir}",
    "--ground-truth", "{ground_truth}", "--validate-every", "3",
    "--device", "cuda", "--epochs", "15", "--samples-per-image", "2",
    "--batch-size", "8", "--stride", "256", "--infer-batch", "16",
    "--selection-max-instances", "4", "--threshold", "0.4", "--threshold-grid", "0.2", "0.3", "0.35", "0.4", "0.5", "0.6",
    "--min-area", "80", "--max-candidates", "20",
])
subprocess.run([
    sys.executable, "main.py", "--data-root", str(data_root), "--run-dir", str(run_dir),
    "--confidence-grid", "0.20", "0.25", "0.30", "0.35", "0.40", "0.50",
    "--max-instances-grid", "1", "2", "3", "4", "5", "8", "10",
    "--model-command", model_command,
], check=True)


In [ ]:
import json

subprocess.run([
    sys.executable, "scripts/audit_submission.py",
    "--submission", str(run_dir / "submission.csv"),
    "--test-images", str(data_root / "test" / "test_images"),
], check=True)
print("Selected model:", json.loads((run_dir / "selected_model.json").read_text()))
print("Selected postprocessing:", json.loads((run_dir / "selected_confidence.json").read_text()))
print("Submission:", run_dir / "submission.csv")
